In [2]:
import shutil
import pandas as pd
from pathlib import Path
from datetime import datetime

In [3]:
# Paths
BASE_DIR     = Path(r"c:\Users\Usuario\Documents\Monroe County ORRI")

# Source folders that contain OH... subfolders (searched recursively)
SOURCE_HG1   = BASE_DIR / "_HG ORRI (1)"
SOURCE_HG2   = BASE_DIR / "_HG ORRI (2.0)"
SOURCE_UTIC  = BASE_DIR / "UTIC files (renamed)"   # top-level subfolders only

# Output
DEST_DIR     = BASE_DIR / "UltimateFiles"
OTHERS_DIR   = DEST_DIR / "others"
REPORT_CSV   = BASE_DIR / "ultimate_report.csv"

print(f"HG ORRI (1)        : {SOURCE_HG1}")
print(f"HG ORRI (2.0)      : {SOURCE_HG2}")
print(f"UTIC files (renamed): {SOURCE_UTIC}")
print(f"Destination         : {DEST_DIR}")
print(f"Others              : {OTHERS_DIR}")

HG ORRI (1)        : c:\Users\Usuario\Documents\Monroe County ORRI\_HG ORRI (1)
HG ORRI (2.0)      : c:\Users\Usuario\Documents\Monroe County ORRI\_HG ORRI (2.0)
UTIC files (renamed): c:\Users\Usuario\Documents\Monroe County ORRI\UTIC files (renamed)
Destination         : c:\Users\Usuario\Documents\Monroe County ORRI\UltimateFiles
Others              : c:\Users\Usuario\Documents\Monroe County ORRI\UltimateFiles\others


In [4]:
# Collect all OH... folders from the three sources
# Each entry: (folder_path, source_label)
oh_candidates = []

# _HG ORRI (1) and (2.0) – search recursively for folders whose name starts with OH
for source, label in [(SOURCE_HG1, "_HG ORRI (1)"), (SOURCE_HG2, "_HG ORRI (2.0)")]:
    for folder in source.rglob("*"):
        if folder.is_dir() and folder.name.upper().startswith("OH"):
            oh_candidates.append((folder, label))

# UTIC files (renamed) – only top-level subfolders
for folder in sorted(SOURCE_UTIC.iterdir()):
    if folder.is_dir() and folder.name.upper().startswith("OH"):
        oh_candidates.append((folder, "UTIC files (renamed)"))

print(f"Total OH folders found across all sources: {len(oh_candidates)}")
for path, label in oh_candidates[:10]:
    print(f"  [{label}] {path.name}")

Total OH folders found across all sources: 558
  [_HG ORRI (1)] OH00143-02
  [_HG ORRI (1)] OH00145-00
  [_HG ORRI (1)] OH00146-00
  [_HG ORRI (1)] OH00147-00
  [_HG ORRI (1)] OH00148-00
  [_HG ORRI (1)] OH00150-00
  [_HG ORRI (1)] OH00151-00
  [_HG ORRI (1)] OH00152-00
  [_HG ORRI (1)] OH00153-00
  [_HG ORRI (1)] OH00154-00


In [5]:
# Create UltimateFiles/ and others/ (fresh start each run)
if DEST_DIR.exists():
    print(f"Removing existing destination: {DEST_DIR}")
    shutil.rmtree(DEST_DIR)

DEST_DIR.mkdir()
OTHERS_DIR.mkdir()
print(f"Created: {DEST_DIR}")
print(f"Created: {OTHERS_DIR}")

Created: c:\Users\Usuario\Documents\Monroe County ORRI\UltimateFiles
Created: c:\Users\Usuario\Documents\Monroe County ORRI\UltimateFiles\others


In [6]:
# Copy OH folders into UltimateFiles/
# If the same folder name appears in multiple sources, suffix with _source label to avoid overwriting
report_rows = []
seen_names = {}   # name -> count, to detect duplicates

for src_path, label in oh_candidates:
    name = src_path.name
    dest_path = DEST_DIR / name

    # Handle name collision
    if dest_path.exists():
        seen_names[name] = seen_names.get(name, 1) + 1
        safe_name = f"{name}__{label.replace(' ', '_')}__{seen_names[name]}"
        dest_path = DEST_DIR / safe_name
        status = f"COPIED (renamed to avoid collision -> {safe_name})"
        print(f"  [DUP]   {name} from [{label}] -> {safe_name}")
    else:
        seen_names[name] = 1
        status = "COPIED"
        print(f"  [OK]    {name} from [{label}]")

    shutil.copytree(src_path, dest_path)
    report_rows.append({
        "folder_name"  : name,
        "dest_name"    : dest_path.name,
        "source"       : label,
        "source_path"  : str(src_path),
        "dest_path"    : str(dest_path),
        "status"       : status,
        "timestamp"    : datetime.now().isoformat(timespec="seconds")
    })

print(f"\nDone. {len(report_rows)} OH folders copied to UltimateFiles/.")

  [OK]    OH00143-02 from [_HG ORRI (1)]
  [OK]    OH00145-00 from [_HG ORRI (1)]
  [OK]    OH00146-00 from [_HG ORRI (1)]
  [OK]    OH00147-00 from [_HG ORRI (1)]
  [OK]    OH00148-00 from [_HG ORRI (1)]
  [OK]    OH00150-00 from [_HG ORRI (1)]
  [OK]    OH00151-00 from [_HG ORRI (1)]
  [OK]    OH00152-00 from [_HG ORRI (1)]
  [OK]    OH00153-00 from [_HG ORRI (1)]
  [OK]    OH00154-00 from [_HG ORRI (1)]
  [OK]    OH00155-00 from [_HG ORRI (1)]
  [OK]    OH00156-00 from [_HG ORRI (1)]
  [OK]    OH00157-00 from [_HG ORRI (1)]
  [OK]    OH00158-00 from [_HG ORRI (1)]
  [OK]    OH00159-00 from [_HG ORRI (1)]
  [OK]    OH00160-00 from [_HG ORRI (1)]
  [OK]    OH00161-00 from [_HG ORRI (1)]
  [OK]    OH00185-00 from [_HG ORRI (1)]
  [OK]    OH00190-00 from [_HG ORRI (1)]
  [OK]    OH00482-00 from [_HG ORRI (1)]
  [OK]    OH00187-00 from [_HG ORRI (1)]
  [OK]    OH00200-00 from [_HG ORRI (1)]
  [OK]    OH00201-00 from [_HG ORRI (1)]
  [OK]    OH00202-00 from [_HG ORRI (1)]
  [OK]    OH0048

In [7]:
# Copy non-OH top-level folders from UTIC files (renamed) into others/
others_rows = []

for folder in sorted(SOURCE_UTIC.iterdir()):
    if not folder.is_dir():
        continue
    if folder.name.upper().startswith("OH"):
        continue   # already handled above

    dest_path = OTHERS_DIR / folder.name

    if dest_path.exists():
        status = "SKIPPED - already exists in others/"
        print(f"  [SKIP]  {folder.name}")
    else:
        shutil.copytree(folder, dest_path)
        status = "COPIED to others/"
        print(f"  [--]    {folder.name} -> others/")

    others_rows.append({
        "folder_name"  : folder.name,
        "dest_name"    : folder.name,
        "source"       : "UTIC files (renamed)",
        "source_path"  : str(folder),
        "dest_path"    : str(dest_path),
        "status"       : status,
        "timestamp"    : datetime.now().isoformat(timespec="seconds")
    })

print(f"\n{len(others_rows)} unallocated folders copied to others/.")

  [--]    UTIC00074-000 -> others/
  [--]    UTIC01038-009 -> others/
  [--]    UTIC01038-010 -> others/
  [--]    UTIC01476-000 -> others/
  [--]    UTIC01488-000 -> others/
  [--]    UTIC01494-000 -> others/
  [--]    UTIC01498-000 -> others/
  [--]    UTIC01501-000 -> others/
  [--]    UTIC01546-000 -> others/
  [--]    UTIC01620-000 -> others/
  [--]    UTIC01628-000 -> others/
  [--]    UTIC01716-000 -> others/
  [--]    UTIC01728-000 -> others/
  [--]    UTIC01753-000 -> others/
  [--]    UTIC01792-000 -> others/
  [--]    UTIC01800-000 -> others/
  [--]    UTIC01810-000 -> others/
  [--]    UTIC01846-000 -> others/
  [--]    UTIC01917-000 -> others/
  [--]    UTIC01934-000 -> others/
  [--]    UTIC01975-000 -> others/
  [--]    UTIC01985-000 -> others/
  [--]    UTIC02001-000 -> others/
  [--]    UTIC02124-000 -> others/
  [--]    UTIC02154-000 -> others/
  [--]    UTIC02355-000 -> others/
  [--]    UTIC02361-000 -> others/
  [--]    UTIC02370-000 -> others/
  [--]    UTIC02436-

In [8]:
# Save combined report CSV
all_rows = report_rows + others_rows
report_df = pd.DataFrame(all_rows)
report_df.to_csv(REPORT_CSV, index=False)
print(f"Report saved -> {REPORT_CSV}")
report_df

Report saved -> c:\Users\Usuario\Documents\Monroe County ORRI\ultimate_report.csv


,folder_name,dest_name,source,source_path,dest_path,status,timestamp
0,OH00143-02,OH00143-02,_HG ORRI (1),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED,2026-03-17T12:19:18
1,OH00145-00,OH00145-00,_HG ORRI (1),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED,2026-03-17T12:19:18
2,OH00146-00,OH00146-00,_HG ORRI (1),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED,2026-03-17T12:19:18
3,OH00147-00,OH00147-00,_HG ORRI (1),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED,2026-03-17T12:19:18
4,OH00148-00,OH00148-00,_HG ORRI (1),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED,2026-03-17T12:19:18
...,...,...,...,...,...,...,...
608,UTIC06740-000,UTIC06740-000,UTIC files (renamed),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED to others/,2026-03-17T12:19:25
609,UTIC07080-000,UTIC07080-000,UTIC files (renamed),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED to others/,2026-03-17T12:19:25
610,UTIC07579-000,UTIC07579-000,UTIC files (renamed),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED to others/,2026-03-17T12:19:25
611,UTIC08307-000,UTIC08307-000,UTIC files (renamed),c:\Users\Usuario\Documents\Monroe County ORRI\...,c:\Users\Usuario\Documents\Monroe County ORRI\...,COPIED to others/,2026-03-17T12:19:25
